# 策略优化导论

本章通过数值实验理解 Policy Optimization（策略优化）。我们使用 Gymnasium 环境，先手工设计策略，再把策略参数当作黑盒优化变量，用进化策略自动搜索。重点是建立直观认识，因此理论推导保持在必要程度。

推荐参考：Richard S. Sutton and Andrew G. Barto, *Reinforcement Learning: An Introduction*, 2nd ed.


## 1. 环境、状态、动作与策略

以 `CartPole-v1` 为例，目标是让小车上的杆尽可能长时间保持竖直。环境给出的观测是一个 4 维向量：

1. cart position：小车位置；
2. cart velocity：小车速度；
3. pole angle：杆的角度；
4. pole angular velocity：杆的角速度。

动作只有两个：向左施力或向右施力。

**策略（policy）** 是从当前观测状态到动作的规则。随机策略可以写成 $a\sim\pi(\cdot\mid s)$，确定性策略可以写成 $a=\mu(s)$。


In [ ]:
# 在 Colab 中可按需安装
# !pip install gymnasium[classic-control]
import random
import numpy as np
import gymnasium as gym


## 2. Rollout：让策略与环境交互

一次 rollout 从环境初始化开始，反复执行“观察状态 → 策略选择动作 → 环境转移到下一状态并给出奖励”，直到 episode 结束。


In [ ]:
def rollout(envname, policy=None, render=False, seed=None):
    env = gym.make(envname, render_mode='rgb_array' if render else None)
    if seed is not None:
        random.seed(int(seed))
    env_seed = random.randint(0, 100000)
    action_seed = random.randint(0, 100000)
    observation, info = env.reset(seed=env_seed)
    env.action_space.seed(action_seed)

    history = []
    images = []
    terminated = truncated = False
    if render:
        images.append(env.render())

    while not (terminated or truncated):
        action = env.action_space.sample() if policy is None else policy(observation)
        next_observation, reward, terminated, truncated, info = env.step(action)
        history.append([observation, action, next_observation, reward, terminated, truncated, info])
        observation = next_observation
        if render:
            images.append(env.render())
    env.close()
    return history, images

def cumulative_reward(history):
    return sum(step[3] for step in history)


## 3. 随机策略作为基线

如果 `rollout` 不传入策略，就从动作空间随机采样动作。这给出了最简单的基线。对于 CartPole，随机策略通常很快失败。


In [ ]:
history, _ = rollout('CartPole-v1', policy=None, seed=0)
print('随机策略累计回报:', cumulative_reward(history))


## 4. 手工设计一个策略

最简单的思路是根据杆的角度决定向哪一侧施力。更进一步，可以同时使用杆角度与角速度。通过手工调参数，我们已经在进行一种低维策略优化。


In [ ]:
def hand_policy(observation):
    position, velocity, angle, angular_velocity = observation
    score = angle + 0.25 * angular_velocity
    return 1 if score > 0 else 0

history, _ = rollout('CartPole-v1', policy=hand_policy, seed=0)
print('手工策略累计回报:', cumulative_reward(history))


## 5. 参数化策略

为了让优化算法自动搜索策略，把规则写成带参数 $\theta$ 的函数。例如线性策略

$$z=\theta^T s,\qquad a=\mathbf{1}[z>0].$$

这样“寻找好的控制规则”就转化为“寻找好的参数向量 $\theta$”。


In [ ]:
def make_linear_policy(theta):
    theta = np.asarray(theta)
    def policy(observation):
        return int(np.dot(theta, observation) > 0.0)
    return policy

theta = np.array([0.0, 0.0, 1.0, 0.25])
history, _ = rollout('CartPole-v1', make_linear_policy(theta), seed=0)
print(cumulative_reward(history))


## 6. 策略评价为什么有噪声

同一个策略在不同初始状态和随机环境下会得到不同回报。因此目标函数并不是简单的确定性函数 $J(\theta)$。更合理的目标是最大化期望回报：

$$J(\theta)=\mathbb{E}[G\mid\pi_\theta].$$

实际中只能用有限次 rollout 的样本均值近似这个期望。评价次数越多，噪声越小，但计算成本越高。


In [ ]:
def evaluate(theta, seeds=range(5)):
    policy = make_linear_policy(theta)
    returns = []
    for seed in seeds:
        history, _ = rollout('CartPole-v1', policy=policy, seed=seed)
        returns.append(cumulative_reward(history))
    return np.mean(returns), np.std(returns)

print(evaluate(theta))


## 7. 把策略优化看作黑盒优化

如果我们只把“参数 $\theta$ → 多次 rollout 的平均回报”当成一个可查询函数，而不利用环境动力学、价值函数或策略梯度信息，那么这就是黑盒策略优化。

进化策略（Evolution Strategy）特别适合这一设置：

- 不需要目标函数可微；
- 只依赖候选解的性能排序或函数值；
- 可以并行评价多个策略；
- 对模拟器、离散逻辑和复杂策略结构都比较自然。


## 8. 使用 CMA-ES 搜索策略参数

CMA-ES 在参数空间中维护一个多元高斯搜索分布，反复执行：

1. 从当前搜索分布采样策略参数；
2. 在环境中评价每个策略；
3. 根据结果更新分布均值、协方差和步长；
4. 重复直到达到预算或性能要求。

对策略优化来说，目标通常定义为平均累计回报的负值，因为标准优化器通常执行最小化。


In [ ]:
# 如果安装了 pycma，可以直接尝试：
# !pip install cma
# import cma
# def objective(theta):
#     mean_return, _ = evaluate(theta, seeds=range(5))
#     return -mean_return
# result = cma.fmin(objective, np.zeros(4), 1.0)


## 9. 黑盒策略优化的优点与限制

优点：实现简单、对不可微系统友好、容易把整个闭环性能作为目标。

限制：每次候选参数都需要完整 rollout，样本效率通常较低；如果策略参数维度很高，搜索会明显变难；它没有利用每一步状态、动作、奖励中蕴含的结构信息。

后续 `2_policy_gradient.ipynb` 会利用轨迹中的状态—动作—奖励结构推导 Policy Gradient，这也是从“黑盒优化”进入“强化学习”的关键转折。


## 本章要点

- Policy 是根据状态选择动作的规则。
- Rollout 是策略与环境闭环交互得到的一条轨迹。
- 策略参数化后，可以直接把期望累计回报作为优化目标。
- CMA-ES 等进化策略可以在完全不知道环境梯度的情况下优化策略。
- 黑盒方法简单但样本效率较低；Policy Gradient 将进一步利用轨迹内部信息。

配套练习见 `ex1_evolutionary_policy_optimization.ipynb`。
